Запустите следующую ячейку и в поле для ввода **впишите номера аудиозаписей**, которые хотите обработать, каждый на новой строке (файлы должны называться "audio" + номер и лежать в папке audio. Формат аудиозаписей - wav). Когда закончите, нажмите Enter.
Например, если хотите обработать **audioS38**, введите просто **S38**, если хотите обработать audio00035, введите 00035.

In [ ]:
import os
files = []
num = input()
while num != '':
    if os.path.exists(f'audio/audio{num}.wav'):
        files.append(num)
    else:
        print('Ошибка: не найден файл audio/audio{num}.wav.\n(Вы можете продолжать вводить, но последняя строка не засчитана)')
    num = input()
with open('audio_filenames.txt', 'w') as f:
    f.write('\n'.join(files))

## GigaAM from GitHub

### Installing reqs and testing

In [ ]:
! git clone https://github.com/salute-developers/GigaAM.git
%cd GigaAM

Cloning into 'GigaAM'...
remote: Enumerating objects: 282, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 282 (delta 67), reused 36 (delta 36), pack-reused 178 (from 1)
Receiving objects: 100% (282/282), 2.26 MiB | 6.58 MiB/s, done.
Resolving deltas: 100% (126/126), done.
/content/GigaAM


In [ ]:
! pip install -e .[tests]

Obtaining file:///content/GigaAM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.6/254.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.9 MB/s eta 0:00:00
  Building editable for gigaam (pyproject.toml) ... done
  Created wheel for gigaam: filename=gigaam-0.1.0-0.editable-py3-none-any.whl size=7974 sha256=6b45f86111647ea85d7cccf09858a41be100e9480e55ec85dcc75951c5ca2ed1
  Stored in directory: /tmp/pip-ephem-wheel-cache-x17_m5pl/wheels/

### Longform

As `.transcribe` function input lenght is limited by 25 seconds, we need `.transcribe_longform` with audio segmentation based on `pyannote/segmentation-3.0` voice activity detection pipeline.

In [ ]:
! pip install pyannote.audio==4.0

In [ ]:
# in colab the session might need restarting here
%cd GigaAM

/content/GigaAM


In the next 2 cells you need a hugging face token with access to model https://huggingface.co/pyannote/segmentation-3.0 (you need to create a HF token, then go to model page and ask for access)

In [ ]:
! HF_TOKEN="" pytest -v tests/test_longform.py --disable-warnings

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/GigaAM
configfile: pyproject.toml
plugins: hydra-core-1.3.2, cov-7.1.0, langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 7 items                                                              

tests/test_longform.py::test_segmentation_functionality[30.0] PASSED     [ 14%]
tests/test_longform.py::test_segmentation_functionality[60.0] PASSED     [ 28%]
tests/test_longform.py::test_segmentation_functionality[120.0] PASSED    [ 42%]
tests/test_longform.py::test_transcribe_longform[v3_ctc] PASSED          [ 57%]
tests/test_longform.py::test_transcribe_longform[v3_e2e_rnnt] PASSED     [ 71%]
tests/test_longform.py::test_longform_consistency[v3_ctc] PASSED         [ 85%]
tests/test_longform.py::test_segmentation_edge_cases PASSED              [100%]

=================== 7 passed, 1 war

In [ ]:
import os
import warnings
warnings.simplefilter("ignore")

import gigaam
import json
os.environ["HF_TOKEN"] = ""


In [ ]:
def word_dict(word):
    return {"text": word.text, "start": word.start, "end": word.end}

In [ ]:
def format(result):
    result_formatted = {'text': str(result), 'segments': [
        {"id": 0, "start": result.words[0].start, "end": result.words[-1].end,
        "words": list([word_dict(i) for i in result.words])}
        ]}
    return result_formatted

In [ ]:
option = "e2e_ctc"
model2 = gigaam.load_model(f"v3_{option}")

100%|███████████████████████████████████████| 422M/422M [00:25<00:00, 17.4MiB/s]
100%|████████████████████████████████████████| 235k/235k [00:00<00:00, 463kiB/s]


In [4]:
filenames = []
with open('audio_filenames.txt') as f:
    filenames = [i.strip() for i in f.readlines()]

['0075']


In [ ]:
for num in filenames:
    long_audio_path = f"../audio/audio{num}.wav"
    result2 = model2.transcribe_longform(long_audio_path, word_timestamps=True)
    with open(f"../transcripts/transcript_{num}_{option}.json", "w") as f:
        f.write(json.dumps(format(result2), indent = 2, ensure_ascii = False))

filtered by duration: 0/5 samples (0.0%), 0.00/0.02 h (0.0%)
